# Comparison of the four error-model configurations — sea-level-contribution learning curves

The sea-level analogue of `Config_Comparison_Parametric_Learning.ipynb`. One panel per projection
horizon (2100/2200/2300): the across-trajectory **mean 90% credible-interval width (mm)** of the
projected sea-level contribution versus analysis year, for all four error-model configurations
(A, B, C, D).

Styling matches the parametric figure (`science` style, same four-configuration colour/linestyle scheme).
**To restyle, edit the `EDIT HERE` block below** (`FONTSIZE`, `FIGSIZE`, `YTOP_FAC`, `SCEN`).

Run from the `notebooks/` directory, all cells top to bottom. It reads
`outputs/projections_<tag>/slr_learning_widths.jld2` (one per configuration, produced by
`Amery_Future_Learning_Plots.ipynb`) and writes
`outputs/plots/Config_Comparison_Plots/sle_learning_all_configs.pdf`. The figure is also shown
inline. Note the four 2015 baselines are configuration-conditional (A/B code-on, C/D code-off) but nearly
identical, so the curves start together and then diverge as each contracts.


In [ ]:
using JLD2, Statistics, PyCall, PyPlot, Printf
# matplotlib needs the conda C++ runtime; harmless no-op outside a conda env.
if haskey(ENV, "CONDA_PREFIX")
    using Libdl
    for lib in ("libstdc++.so.6", "libgcc_s.so.1")
        p = joinpath(ENV["CONDA_PREFIX"], "lib", lib); isfile(p) && Libdl.dlopen(p, Libdl.RTLD_GLOBAL)
    end
end

const OUTDIR = "../outputs/plots/Config_Comparison_Plots"; mkpath(OUTDIR)
try; pyimport("scienceplots"); PyPlot.matplotlib.pyplot.style.use(["default","science","no-latex"]); catch; end

# ================================ EDIT HERE =====================================
FONTSIZE = 26.0                                   # master font size
FIGSIZE  = (32, 6.15)                             # (width, height) inches — controls the aspect ratio
YTOP_FAC = 1.2                                    # y-axis headroom above the tallest 2015 baseline
# one row per configuration: (legend label, projections tag, colour, linestyle)
SCEN = [("A (independent, code on)",  "indep_code",   "#1f5fa8", "--"),
        ("B (Wiener, code on)",       "wiener_code",  "#c0392b", "--"),
        ("C (independent, code off)", "indep_nocode", "#1f5fa8", "-"),
        ("D (Wiener, code off)",      "wiener_nocode","#c0392b", "-")]
# ================================================================================

PRED = [2100, 2200, 2300]                         # projection horizons (one panel each)
cal_range(py) = collect(range(2015, step=15, length=fld(py - 2015, 15) + 1))   # analysis years <= py
xticks_dict = Dict(2100=>[2015,2030,2045,2060,2075,2090],
                   2200=>[2015,2045,2075,2105,2135,2165,2195],
                   2300=>[2015,2050,2100,2150,2200,2250,2300])
yticks_dict     = Dict(2100=>[1.5,3,4.5,6,7.5], 2200=>[12,24,36,48,60,72], 2300=>[14,28,42,56,70,84])
letters = ["a","b","c"]
colmean(W) = [ (v = filter(!isnan, W[:, k]); isempty(v) ? NaN : mean(v)) for k in 1:size(W, 2) ];


In [ ]:
# across-trajectory MEAN of the 90% CI width (mm): MW[tag][py] = vector over analysis years
MW = Dict{String,Dict{Int,Vector{Float64}}}()
for (_, tag, _, _) in SCEN
    d = JLD2.load("../outputs/projections_$(tag)/slr_learning_widths.jld2", "slr_widths")
    MW[tag] = Dict(py => colmean(d[py]) for py in PRED)
end
println("Per-configuration mean 90% CI width (mm): baseline(2015) -> final(horizon)")
for py in PRED, (lab, tag, _, _) in SCEN
    m = MW[tag][py]; @printf("  %-22s SLE %d : %8.3f -> %7.3f   (fold %.1fx)\n", lab, py, m[1], m[end], m[1]/m[end])
end

In [ ]:
fig, axs = PyPlot.subplots(nrows=1, ncols=3, figsize=FIGSIZE, dpi=300, gridspec_kw=Dict("wspace"=>0.14))

for (a, py) in enumerate(PRED)
    ax = axs[a]; years = cal_range(py)
    for (lab, tag, col, ls) in SCEN
        ax.plot(years, MW[tag][py], color=col, linestyle=ls, lw=2.4, marker="o", ms=5, label=lab)
    end
    ax.set_xlabel("Analysis year", fontsize=FONTSIZE*0.82, labelpad=10)
    a == 1 && ax.set_ylabel("90% credible interval width (mm)", fontsize=FONTSIZE*0.82, labelpad=12)
    ax.set_title("Sea-level contribution at $py", fontsize=1.05*FONTSIZE, pad=10)
    ax.set_ylim(0.0, YTOP_FAC * maximum(MW[tag][py][1] for (_, tag, _, _) in SCEN))   # tallest 2015 baseline
    ax.set_xlim(2015, years[end]); ax.set_xticks(xticks_dict[py]); ax.set_yticks(yticks_dict[py])
    ax.tick_params(axis="both", which="major", labelsize=0.72*FONTSIZE)
    ax.grid(true, alpha=0.5, zorder=1)
    ax.text(a == 1 ? -0.13 : -0.1, 1.08, "($(letters[a]))", ha="left", va="top",
            transform=ax.transAxes, fontsize=FONTSIZE)
end

h, l = axs[1].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=4, fontsize=0.78*FONTSIZE, frameon=true, framealpha=1,
           bbox_to_anchor=(0.5, 0.955), bbox_transform=fig.transFigure)

out = joinpath(OUTDIR, "sle_learning_all_configs.pdf")
fig.savefig(out; bbox_inches="tight"); println("Wrote ", out)